# Employee Attrition Audit — single-notebook experiment

**Authors:** Kritarth Prasad, Divyansh Audichya, Pranjal Gaur, Tanay Patel, Md Safwan Areeb

This notebook is the whole project. It will contain, in order:

1. **Data audit and B0 anchors** — confirm the datasets and the no-skill numbers.
2. **B1 — protocol-faithful replication** — SMOTE before the split, provenance tracking, real vs synthetic test rows.
3. **B2 — leakage-safe baseline** — split first, SMOTE inside each fold, repeated five-fold CV, full metric suite.
4. **B3 — deployment audit** — calibration, decision thresholds, subgroup fairness, bidirectional transfer, SHAP.

## Data files expected

Put the raw files here and do not edit them:

- `data/ibm_hr.csv` — Kaggle IBM HR Analytics: 1,470 rows, 35 columns, 16.12% attrition.
- `data/saudi_attrition.csv` — Mendeley Saudi Employee Attrition: 1,174 cleaned rows, 42.42% attrition.

If your downloaded IBM file is named `WA_Fn-UseC_-HR-Employee-Attrition.csv`, the loader will find it too.

## Rules

- Keep B1 and B2 as separate code paths. B1 is intentionally flawed; B2 is the clean protocol.
- Freeze seeds and target decisions before opening the hold-out.
- Never tune on the test set.
- A blank or missing value never means zero.

## Model roster

| Stage | Model | Role |
|---|---|---|
| B0 | Majority class | No-skill anchor |
| B1 | Random Forest | Paper roster |
| B1 | SVM (RBF) | Paper roster |
| B1 | Decision Tree | Paper roster |
| B1 | Gradient Boosting | Paper roster |
| B1 | Hard-voting hybrid | Paper's claimed best model |
| B2 | Random Forest | Same learner, clean protocol |
| B2 | SVM (RBF) | Same learner, clean protocol |
| B2 | Decision Tree | Same learner, clean protocol |
| B2 | Gradient Boosting | Same learner, clean protocol |
| B2 | Logistic Regression | Linear benchmark; stacking meta-learner |
| B2 | Extra Trees | Strong tabular learner; corpus comparison |
| B2 | XGBoost | Credible frontier model; TreeSHAP-compatible |
| B2 | LightGBM | Credible frontier model; corpus comparison |
| B2 | AdaBoost | Best model in AL-Ali et al.; replication check |
| Ensemble | Hard voting | Paper's rule |
| Ensemble | Soft voting | Probability averaging |
| Ensemble | Weighted voting | Weights from out-of-fold PR-AUC |
| Ensemble | Stacking | Logistic-regression meta-learner |

The four ensembles use the paper's four base learners only, so the combination rule is the only thing that changes.


In [ ]:
# Core imports and fixed configuration.
import os
import sys
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_validate,
    cross_val_predict,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Freeze these values before running B2/B3. Do not change after
# looking at the hold-out.
# ------------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
CV_REPEATS = 5
SMOTE_K = 5
SMOTE_RATIO = 1.0

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)


## 1. Data audit and B0 anchors

This section does two things:

1. Loads the IBM and Saudi files with explicit target coding.
2. Computes the majority-class baseline.

The B0 check is the first real test of the notebook. If the B0 numbers do not match the arithmetic anchors, the data loader or metric code is wrong, and no later result can be trusted.

Expected:

| Dataset | Accuracy | Balanced accuracy | Recall | Macro F1 | PR-AUC | Brier |
|---|---:|---:|---:|---:|---:|---:|
| IBM | 83.88% | 50.0% | 0.0% | 45.62% | 16.12% | 0.135 |
| Saudi | 57.58% | 50.0% | 0.0% | 36.54% | 42.42% | 0.244 |

Note: the majority class has recall 0 and F1 0, but the negative class has a nonzero F1. Macro F1 is therefore **not** 50%; it is 45.62% on IBM and 36.54% on Saudi.


In [ ]:
IBM_CANDIDATES = [
    DATA_DIR / "ibm_hr.csv",
    DATA_DIR / "WA_Fn-UseC_-HR-Employee-Attrition.csv",
    DATA_DIR / "IBM-HR-Analytics-Employee-Attrition-and-Performance.csv",
]
SAUDI_CANDIDATES = [
    DATA_DIR / "saudi_attrition.csv",
    DATA_DIR / "saudi_employee_attrition.csv",
    DATA_DIR / "employee_attrition_saudi.csv",
]


def find_first(candidates):
    for path in candidates:
        if Path(path).exists():
            return Path(path)
    return None


def read_csv_with_fallback(path):
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not read {path} with utf-8, utf-8-sig, or latin-1.")


def load_ibm(path):
    df = read_csv_with_fallback(path)
    if "Attrition" not in df.columns:
        raise ValueError("IBM file must contain an 'Attrition' column.")

    df = df.copy()
    y = df["Attrition"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
    if y.isna().any():
        bad = df.loc[y.isna(), "Attrition"].unique()[:5]
        raise ValueError(f"Unexpected Attrition values: {bad}")

    drop = [c for c in ("EmployeeNumber", "EmployeeCount", "Over18", "StandardHours") if c in df.columns]
    X = df.drop(columns=drop + ["Attrition"])
    y = y.astype(int)

    meta = {
        "path": str(path),
        "rows": len(df),
        "features": X.shape[1],
        "positives": int(y.sum()),
        "prevalence_pct": round(100 * y.mean(), 2),
    }
    return X, y, meta


def load_saudi(path):
    df = read_csv_with_fallback(path)
    target_candidates = {
        "attrition", "turnover", "leave", "left", "resigned", "jobchange",
        "job_change", "target", "status",
    }
    target = next((c for c in df.columns if c.strip().lower() in target_candidates), None)
    if target is None:
        raise ValueError(
            "Could not identify the Saudi target column. "
            f"Columns are: {list(df.columns)}"
        )

    df = df.copy()
    raw = df[target]
    if raw.dtype == object:
        y = raw.astype(str).str.strip().str.lower().map({
            "yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0,
        })
    else:
        y = pd.to_numeric(raw, errors="coerce")

    if y.isna().any():
        raise ValueError(f"Unmapped target values in {target}: {raw[y.isna()].unique()[:5]}")

    id_like = {"id", "employeeid", "employee id", "employee_id", "respondent id", "respondent_id"}
    drop = [target] + [c for c in df.columns if c.strip().lower() in id_like]
    X = df.drop(columns=drop, errors="ignore")

    # Drop constant columns and columns that are entirely missing.
    keep = [c for c in X.columns if X[c].notna().any() and X[c].nunique(dropna=False) > 1]
    X = X[keep]
    y = y.astype(int)

    meta = {
        "path": str(path),
        "rows": len(df),
        "features": X.shape[1],
        "positives": int(y.sum()),
        "prevalence_pct": round(100 * y.mean(), 2),
    }
    return X, y, meta


def b0_scores(y):
    y = np.asarray(y).astype(int)
    pred = np.zeros_like(y)
    prevalence = float(y.mean())
    prob = np.full(len(y), prevalence, dtype=float)

    return pd.Series({
        "accuracy_pct": 100 * accuracy_score(y, pred),
        "balanced_accuracy_pct": 100 * balanced_accuracy_score(y, pred),
        "precision_pct": 100 * precision_score(y, pred, zero_division=0),
        "recall_pct": 100 * recall_score(y, pred, zero_division=0),
        "f1_pct": 100 * f1_score(y, pred, zero_division=0),
        "macro_f1_pct": 100 * f1_score(y, pred, average="macro", zero_division=0),
        "roc_auc_pct": 100 * roc_auc_score(y, prob),
        "pr_auc_pct": 100 * average_precision_score(y, prob),
        "mcc": matthews_corrcoef(y, pred),
        "brier": brier_score_loss(y, prob),
    }, name="B0 majority class")


def show_b0(name, y):
    scores = b0_scores(y)
    print(f"\n{name}: n={len(y)}, positives={int(np.sum(y))}, prevalence={100*y.mean():.2f}%")
    print(scores.round(4).to_string())
    return scores


In [ ]:
ibm_path = find_first(IBM_CANDIDATES)
saudi_path = find_first(SAUDI_CANDIDATES)

print("IBM file:", ibm_path)
print("Saudi file:", saudi_path)

if ibm_path is None:
    print("\nIBM data not found. Put the Kaggle CSV at data/ibm_hr.csv.")
else:
    X_ibm, y_ibm, meta_ibm = load_ibm(ibm_path)
    print("\nIBM metadata:", meta_ibm)
    b0_ibm = show_b0("IBM", y_ibm)

if saudi_path is None:
    print("\nSaudi data not found. Put the Mendeley cleaned CSV at data/saudi_attrition.csv.")
else:
    X_saudi, y_saudi, meta_saudi = load_saudi(saudi_path)
    print("\nSaudi metadata:", meta_saudi)
    b0_saudi = show_b0("Saudi", y_saudi)


## 2. Model registry

The registry is deliberately split by stage.

- **B1** uses the paper's four learners with minimal hyperparameter tuning and a hard-voting hybrid.
- **B2** uses the same four learners plus five context learners.
- **Ensembles** use only the four paper learners, so the combination rule is the only change.

The preprocessing object is fitted inside the training partition in B2. For B1, the whole dataset is preprocessed and resampled before the split — that is the audit instrument, not an accident.


In [ ]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric = Pipeline(numeric_steps)
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric, make_column_selector(dtype_include=np.number)),
            ("cat", categorical, make_column_selector(dtype_exclude=np.number)),
        ],
        remainder="drop",
    )


def b1_estimators(seed=RANDOM_STATE):
    # The paper roster. Minimal tuning; the article does not report settings.
    return {
        "RF": RandomForestClassifier(random_state=seed, n_jobs=-1),
        "SVM": CalibratedClassifierCV(SVC(), ensemble=False),
        "DT": DecisionTreeClassifier(random_state=seed),
        "GB": GradientBoostingClassifier(random_state=seed),
    }


def b2_estimators(seed=RANDOM_STATE):
    # The clean comparison roster. Sensible defaults; tune inside training folds later.
    return {
        "RF": RandomForestClassifier(
            n_estimators=500, max_features="sqrt", random_state=seed, n_jobs=-1
        ),
        "SVM": CalibratedClassifierCV(SVC(C=1.0, gamma="scale"), ensemble=False),
        "DT": DecisionTreeClassifier(min_samples_leaf=5, random_state=seed),
        "GB": GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=3, random_state=seed
        ),
        "LR": LogisticRegression(max_iter=5000, random_state=seed),
        "ET": ExtraTreesClassifier(
            n_estimators=500, max_features="sqrt", random_state=seed, n_jobs=-1
        ),
        "XGB": XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1,
        ),
        "LGBM": LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ),
        "ADA": AdaBoostClassifier(
            n_estimators=300, learning_rate=0.05, random_state=seed
        ),
    }


def make_b2_pipeline(name, seed=RANDOM_STATE):
    # Leakage-safe pipeline: preprocessing and SMOTE are fitted only on training rows.
    estimators = b2_estimators(seed)
    if name not in estimators:
        raise KeyError(f"Unknown model {name!r}. Choose from {sorted(estimators)}")
    scale = name in {"SVM", "LR"}
    return ImbPipeline([
        ("prep", make_preprocessor(scale_numeric=scale)),
        ("smote", SMOTE(random_state=seed, k_neighbors=SMOTE_K)),
        ("model", estimators[name]),
    ])


def make_ensemble(kind, seed=RANDOM_STATE):
    # Ensembles over the four paper learners only.
    base = b1_estimators(seed)
    estimators = list(base.items())
    if kind == "hard":
        return VotingClassifier(estimators=estimators, voting="hard")
    if kind == "soft":
        return VotingClassifier(estimators=estimators, voting="soft")
    if kind == "stack":
        return StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(max_iter=5000, random_state=seed),
            cv=CV_FOLDS,
            stack_method="predict_proba",
            n_jobs=-1,
        )
    raise ValueError("kind must be 'hard', 'soft', or 'stack'")


In [ ]:
print("B1 estimators:")
for name, estimator in b1_estimators().items():
    print(f"  {name:4s} {type(estimator).__name__}")

print("\nB2 estimators:")
for name, estimator in b2_estimators().items():
    print(f"  {name:4s} {type(estimator).__name__}")

print("\nB2 leakage-safe pipeline (preview):")
print(make_b2_pipeline("RF"))

print("\nEnsembles use the four paper learners only: RF, SVM, DT, GB.")
print("Weighted voting is intentionally left for the ensemble cell; it needs out-of-fold scores.")


## 3. B1 — protocol-faithful replication

This section deliberately reproduces the **described** protocol, including its flaw.

1. Encode the whole IBM dataset.
2. Apply SMOTE to the whole dataset, not just the training partition.
3. Split the resampled data 85:15.
4. Train RF, SVM, DT, and GB.
5. Take a hard vote.
6. Report accuracy on **all** test rows, **real** test rows, and **synthetic** test rows.
7. Keep each synthetic row's two parent indices so the contamination can be traced.

This is not a model we recommend. It is the audit instrument.

### Why the three accuracies matter

- `acc_all` is the paper-like number: it includes synthetic rows that may be near-copies of training rows.
- `acc_real` removes the synthetic test rows. It is still not a clean estimate, because a real test row can have a synthetic child in training.
- `acc_synth` isolates the interpolation advantage. If it is much higher than `acc_real`, the test set has been made easier.

The key quantity is **`acc_all - acc_real`**. A large gap means the headline accuracy is partly an artifact of the test-set composition.


In [ ]:
# B1 helpers: manual SMOTE with parent tracking, models, metrics, and one run.

def manual_smote(X, y, k=5, ratio=1.0, seed=42):
    '''SMOTE with explicit provenance.

    Returns X_combined, y_combined, is_synthetic, parent1, parent2.
    parent1 and parent2 are row positions in the original X; -1 for real rows.
    '''
    X = np.asarray(X)
    y = np.asarray(y).astype(int)
    rng = np.random.default_rng(seed)

    minority = np.where(y == 1)[0]
    majority_n = int((y == 0).sum())
    target_minority = int(round(ratio * majority_n))
    n_synthetic = max(0, target_minority - len(minority))

    if n_synthetic == 0:
        return (
            X.copy(), y.copy(),
            np.zeros(len(y), dtype=bool),
            np.full(len(y), -1, dtype=int),
            np.full(len(y), -1, dtype=int),
        )

    k_eff = min(k + 1, len(minority))
    nearest = NearestNeighbors(n_neighbors=k_eff).fit(X[minority])
    _, neighbour_positions = nearest.kneighbors(X[minority])

    base_pos = rng.integers(0, len(minority), size=n_synthetic)
    if k_eff > 1:
        choice = rng.integers(1, k_eff, size=n_synthetic)
    else:
        choice = np.zeros(n_synthetic, dtype=int)

    neighbour_pos = neighbour_positions[base_pos, choice]
    lam = rng.random(n_synthetic)

    base_global = minority[base_pos]
    neighbour_global = minority[neighbour_pos]
    synthetic_X = X[base_global] + lam[:, None] * (X[neighbour_global] - X[base_global])

    X_combined = np.vstack([X, synthetic_X])
    y_combined = np.concatenate([y, np.ones(n_synthetic, dtype=int)])
    is_synthetic = np.concatenate([
        np.zeros(len(y), dtype=bool),
        np.ones(n_synthetic, dtype=bool),
    ])
    parent1 = np.concatenate([np.full(len(y), -1, dtype=int), base_global])
    parent2 = np.concatenate([np.full(len(y), -1, dtype=int), neighbour_global])

    return X_combined, y_combined, is_synthetic, parent1, parent2


def b1_models(seed=RANDOM_STATE):
    # The paper's roster. SVM probabilities come from Platt scaling.
    return {
        "RF": RandomForestClassifier(random_state=seed, n_jobs=-1),
        "SVM": Pipeline([
            ("scale", StandardScaler()),
            ("svc", CalibratedClassifierCV(SVC(), ensemble=False)),
        ]),
        "DT": DecisionTreeClassifier(random_state=seed),
        "GB": GradientBoostingClassifier(random_state=seed),
    }


def b1_metric_row(y_true, y_pred, y_prob):
    both_classes = len(np.unique(y_true)) > 1
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, y_prob) if both_classes else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
    }


def run_b1_once(X_encoded, y, seed=42, ratio=1.0, k=5, split="stratified"):
    X_combined, y_combined, is_synthetic, parent1, parent2 = manual_smote(
        X_encoded, y, k=k, ratio=ratio, seed=seed
    )

    indices = np.arange(len(y_combined))
    stratify = y_combined if split == "stratified" else None
    train_idx, test_idx = train_test_split(
        indices, test_size=0.15, random_state=seed, stratify=stratify
    )

    X_train, X_test = X_combined[train_idx], X_combined[test_idx]
    y_train, y_test = y_combined[train_idx], y_combined[test_idx]
    synthetic_test = is_synthetic[test_idx]

    models = b1_models(seed)
    predictions, probabilities = {}, {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_test)
        probabilities[name] = model.predict_proba(X_test)[:, 1]

    vote = np.vstack([predictions[name] for name in models]).sum(axis=0)
    mean_probability = np.vstack([probabilities[name] for name in models]).mean(axis=0)
    hybrid_pred = np.where(
        vote > 2, 1,
        np.where(vote < 2, 0, (mean_probability >= 0.5).astype(int)),
    )

    rows = []
    for name in list(models) + ["HYBRID"]:
        pred = hybrid_pred if name == "HYBRID" else predictions[name]
        prob = mean_probability if name == "HYBRID" else probabilities[name]

        row = b1_metric_row(y_test, pred, prob)
        row["model"] = name
        row["acc_all"] = row.pop("accuracy")
        row["acc_real"] = accuracy_score(y_test[~synthetic_test], pred[~synthetic_test])
        row["acc_synth"] = (
            accuracy_score(y_test[synthetic_test], pred[synthetic_test])
            if synthetic_test.sum() else np.nan
        )
        row["n_test"] = len(test_idx)
        row["n_test_real"] = int((~synthetic_test).sum())
        row["n_test_synth"] = int(synthetic_test.sum())
        row["seed"] = seed
        row["ratio"] = ratio
        row["k"] = k
        row["split"] = split
        rows.append(row)

    return pd.DataFrame(rows), parent1, parent2, is_synthetic


In [ ]:
# Run B1: 30 seeds for the main configuration, 5 seeds for sensitivity checks.
if "X_ibm" not in globals():
    raise RuntimeError("Run the IBM data cell first. B1 needs X_ibm and y_ibm.")

b1_preprocessor = make_preprocessor(scale_numeric=False)
X_ibm_encoded = b1_preprocessor.fit_transform(X_ibm)
y_ibm_array = np.asarray(y_ibm).astype(int)

B1_SEEDS = 30
b1_frames = []
for seed in range(B1_SEEDS):
    frame, *_ = run_b1_once(
        X_ibm_encoded, y_ibm_array,
        seed=seed, ratio=1.0, k=5, split="stratified",
    )
    b1_frames.append(frame)

b1_main = pd.concat(b1_frames, ignore_index=True)
b1_main["gap_real"] = b1_main["acc_all"] - b1_main["acc_real"]

b1_summary = b1_main.groupby("model").agg(
    acc_all_mean=("acc_all", "mean"),
    acc_all_median=("acc_all", "median"),
    acc_all_sd=("acc_all", "std"),
    acc_real_mean=("acc_real", "mean"),
    acc_synth_mean=("acc_synth", "mean"),
    gap_real_mean=("gap_real", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    pr_auc_mean=("pr_auc", "mean"),
).sort_values("acc_all_mean", ascending=False)

print("B1 main configuration: 85:15 split, SMOTE ratio 1:1, k=5, stratified, 30 seeds")
print((b1_summary * 100).round(2).to_string())
print("\nHybrid accuracy percentiles (0, 5, 25, 50, 75, 95, 100):")
print((np.percentile(
    b1_main.loc[b1_main.model == "HYBRID", "acc_all"],
    [0, 5, 25, 50, 75, 95, 100],
) * 100).round(2))

# Sensitivity grid: fewer seeds because the point is the direction, not precision.
sensitivity_frames = []
configs = [
    (0.5, 5, "stratified"),
    (1.0, 3, "stratified"),
    (1.0, 7, "stratified"),
    (1.0, 5, "simple"),
]
for ratio, k, split in configs:
    for seed in range(5):
        frame, *_ = run_b1_once(
            X_ibm_encoded, y_ibm_array,
            seed=seed, ratio=ratio, k=k, split=split,
        )
        sensitivity_frames.append(frame)

b1_sensitivity = pd.concat(sensitivity_frames, ignore_index=True)
b1_sensitivity["gap_real"] = b1_sensitivity["acc_all"] - b1_sensitivity["acc_real"]
b1_sensitivity_summary = (
    b1_sensitivity
    .groupby(["ratio", "k", "split", "model"])
    [["acc_all", "acc_real", "acc_synth", "gap_real"]]
    .mean()
    .reset_index()
)

print("\nB1 sensitivity (mean over 5 seeds; HYBRID and SVM only):")
print(
    (b1_sensitivity_summary[
        b1_sensitivity_summary.model.isin(["HYBRID", "SVM"])
    ] * 1).round(3).to_string(index=False)
)


## 4. What B1 found

These are observed results from the run on the IBM HR Analytics file (`SHA-256 a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659`), 30 seeds, SMOTE ratio 1:1, `k=5`, stratified 85:15 split.

### Main configuration

| Model | Acc all (mean) | Median | SD | Acc real | Acc synth | Gap |
|---|---:|---:|---:|---:|---:|---:|
| SVM | 93.1% | 93.0% | 1.3% | 90.1% | 97.5% | 3.0 pp |
| Hard-vote hybrid | 92.9% | 92.7% | 1.4% | 88.5% | 99.4% | 4.4 pp |
| Random Forest | 92.7% | 92.7% | 1.1% | 88.3% | 99.4% | 4.4 pp |
| Gradient Boosting | 92.4% | 92.7% | 1.2% | 87.8% | 99.4% | 4.7 pp |
| Decision Tree | 86.0% | 85.9% | 1.8% | 78.7% | 96.9% | 7.3 pp |

### Three findings

1. **The 95.0% is not the typical result.**  
   The hybrid’s mean is 92.9%, median 92.7%, and SD 1.4 pp. The 95th percentile is 95.0%; the maximum is 95.4%. The reported 95.0% is therefore at the upper tail of the reconstructed distribution, not the central estimate.

2. **The hard-vote hybrid has no clear advantage.**  
   SVM reaches 93.1% mean accuracy, numerically just above the hybrid’s 92.9%. The difference is well within seed noise. The paper’s claim that the hybrid is clearly best is not supported by this reconstruction; the honest statement is that the hybrid is one of several similar models.

3. **The synthetic rows are much easier than the real rows.**  
   The hybrid scores 99.4% on synthetic test rows and 88.5% on real test rows, a gap of 4.4 pp. This is directionally consistent with leakage, but it is just below the pre-registered 5 pp threshold.

### Hypothesis status

- **H1a** (`B1a ≥ 93%`): **fails narrowly** — mean 92.9%, median 92.7%.
- **H1b** (`B1a − B1b ≥ 5 pp`): **fails narrowly** on the main configuration — 4.4 pp. The simple-random split variant reaches 5.4 pp, so the result is sensitive to the split variant.
- **H1c** (`B1a − B2 ≥ 5 pp`): **not yet tested** — needs B2.

### Sensitivity

| Configuration | Hybrid acc | Hybrid gap | SVM acc |
|---|---:|---:|---:|
| Ratio 0.5, k=5 | 89.7% | 2.0 pp | 88.3% |
| Ratio 1.0, k=3 | 92.8% | 4.4 pp | 92.1% |
| Ratio 1.0, k=7 | 93.0% | 4.1 pp | 92.4% |
| Ratio 1.0, simple split | 92.1% | 5.4 pp | 92.3% |

The qualitative pattern is stable: the hybrid is around 92–93%, and the synthetic-row advantage is positive but not always above 5 pp. A lower SMOTE ratio shrinks the gap because fewer synthetic near-copies enter the test set.

### Interpretation

The headline 95.0% is **not reproducible as a central estimate** under the described protocol. It is reproducible as a high-seed outcome. That is a weaker and more honest claim: the protocol can produce 95%, but it does not reliably do so, and it does so partly by making the test set easier.

The next section builds B2, the leakage-safe baseline, to estimate the same task without the contaminated test set. The comparison `B1a − B2` is the audit’s main quantity.


## 5. B2 — leakage-safe baseline

B2 is the clean counterpart to B1.

- 80:20 stratified hold-out, seed 42.
- The hold-out is not used for model selection.
- Preprocessing and SMOTE are fitted **inside each training fold**.
- Repeated stratified five-fold CV: 5 folds × 5 repeats = 25 fold estimates.
- Nine base learners: RF, SVM, DT, GB, LR, ET, XGBoost, LightGBM, AdaBoost.
- Four ensembles over the four paper learners: hard, soft, weighted, stacking.

The selection rule is fixed before looking at the hold-out:

> Among models with mean CV accuracy at least 85%, choose the highest mean cross-validated PR-AUC.

The 85% floor comes from the B2 target range. It prevents a high-recall, low-accuracy model such as logistic regression from being selected only because its PR-AUC is marginally higher.


In [ ]:
# B2 helpers: metric row and ensemble rules over the four paper learners.

def b2_metric_row(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    both_classes = len(np.unique(y_true)) > 1
    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob) if both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if both_classes else np.nan,
        "mcc": matthews_corrcoef(y_true, pred),
        "brier": brier_score_loss(y_true, prob),
    }


def b2_ensemble_scores(prob_matrix, y_true, paper_idx=(0, 1, 2, 3)):
    # The primary ensembles use the four paper learners only.
    probs = prob_matrix[:, list(paper_idx)]
    m = probs.shape[1]

    soft = probs.mean(axis=1)
    votes = (probs >= 0.5).sum(axis=1)
    hard = np.where(
        votes > m / 2, 1,
        np.where(votes < m / 2, 0, (soft >= 0.5).astype(int)),
    )

    aps = np.array([average_precision_score(y_true, probs[:, j]) for j in range(m)])
    weights = np.maximum(0, aps - y_true.mean())
    weights = np.ones(m) / m if weights.sum() == 0 else weights / weights.sum()
    weighted = probs @ weights

    stack = cross_val_predict(
        LogisticRegression(max_iter=5000),
        probs, y_true,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        method="predict_proba",
    )[:, 1]

    rows = []
    for name, prob in [
        ("HARD", hard),
        ("SOFT", soft),
        ("WEIGHTED", weighted),
        ("STACK", stack),
    ]:
        row = b2_metric_row(y_true, prob)
        if name == "HARD":
            # Hard voting has no probability output; ranking and Brier are not meaningful.
            row["roc_auc"] = np.nan
            row["pr_auc"] = np.nan
            row["brier"] = np.nan
        row["model"] = name
        rows.append(row)
    return rows


In [ ]:
# Run B2: 80:20 hold-out and 5 x 5 repeated stratified CV.
train_pos, test_pos = train_test_split(
    np.arange(len(X_ibm)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_ibm,
)

X_b2_train, y_b2_train = X_ibm.iloc[train_pos], y_ibm.iloc[train_pos]
X_b2_test, y_b2_test = X_ibm.iloc[test_pos], y_ibm.iloc[test_pos]

print(f"B2 training rows: {len(X_b2_train)}; hold-out rows: {len(X_b2_test)}")
print(f"Hold-out positives: {int(y_b2_test.sum())}")

b2_model_names = list(b2_estimators())
b2_cv = RepeatedStratifiedKFold(
    n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=RANDOM_STATE
)

# OOF probabilities are stored per repeat for the ensemble comparison.
b2_oof = {
    r: np.full((len(X_b2_train), len(b2_model_names)), np.nan)
    for r in range(CV_REPEATS)
}

b2_base_rows = []
for fold_number, (tr, va) in enumerate(b2_cv.split(X_b2_train, y_b2_train)):
    repeat = fold_number // CV_FOLDS
    for model_number, name in enumerate(b2_model_names):
        pipe = make_b2_pipeline(name)
        pipe.fit(X_b2_train.iloc[tr], y_b2_train.iloc[tr])
        prob = pipe.predict_proba(X_b2_train.iloc[va])[:, 1]
        b2_oof[repeat][va, model_number] = prob

        row = b2_metric_row(y_b2_train.iloc[va], prob)
        row.update(model=name, repeat=repeat, fold=fold_number % CV_FOLDS)
        b2_base_rows.append(row)

    if (fold_number + 1) % CV_FOLDS == 0:
        print(f"repeat {repeat + 1} complete")

b2_base = pd.DataFrame(b2_base_rows)
b2_base_summary = b2_base.groupby("model").agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_median=("accuracy", "median"),
    accuracy_sd=("accuracy", "std"),
    balanced_mean=("balanced_accuracy", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    macro_f1_mean=("macro_f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    pr_auc_mean=("pr_auc", "mean"),
    mcc_mean=("mcc", "mean"),
    brier_mean=("brier", "mean"),
).sort_values("pr_auc_mean", ascending=False)

print("\nB2 base learners: mean across 25 fold estimates")
print((b2_base_summary * 100).round(2).to_string())

# Ensemble comparison from the OOF probability matrix.
b2_ens_rows = []
for repeat in range(CV_REPEATS):
    for row in b2_ensemble_scores(b2_oof[repeat], y_b2_train.values):
        row["repeat"] = repeat
        b2_ens_rows.append(row)

b2_ens = pd.DataFrame(b2_ens_rows)
b2_ens_summary = b2_ens.groupby("model").agg(
    accuracy_mean=("accuracy", "mean"),
    balanced_mean=("balanced_accuracy", "mean"),
    recall_mean=("recall", "mean"),
    macro_f1_mean=("macro_f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    pr_auc_mean=("pr_auc", "mean"),
    mcc_mean=("mcc", "mean"),
    brier_mean=("brier", "mean"),
).sort_values("pr_auc_mean", ascending=False)

print("\nB2 ensembles: mean across 5 OOF repeats")
print((b2_ens_summary * 100).round(2).to_string())

# Fixed selection rule: accuracy floor 85%, then highest PR-AUC.
base_stats = b2_base.groupby("model").agg(
    accuracy=("accuracy", "mean"), pr_auc=("pr_auc", "mean")
)
ens_stats = b2_ens.groupby("model").agg(
    accuracy=("accuracy", "mean"), pr_auc=("pr_auc", "mean")
)
all_stats = pd.concat([base_stats, ens_stats])
eligible = all_stats[all_stats.accuracy >= 0.85]
b2_selected = eligible["pr_auc"].idxmax()
print("\nEligible models (accuracy >= 85%), sorted by PR-AUC:")
print((eligible * 100).round(2).sort_values("pr_auc", ascending=False).to_string())
print(f"\nSelected B2 model: {b2_selected}")

# Evaluate the selected model once on the untouched hold-out.
paper_names = ["RF", "SVM", "DT", "GB"]
paper_idx = [b2_model_names.index(n) for n in paper_names]
paper_test_probs = np.column_stack([
    make_b2_pipeline(n).fit(X_b2_train, y_b2_train).predict_proba(X_b2_test)[:, 1]
    for n in paper_names
])

if b2_selected in b2_model_names:
    b2_test_prob = (
        make_b2_pipeline(b2_selected)
        .fit(X_b2_train, y_b2_train)
        .predict_proba(X_b2_test)[:, 1]
    )
elif b2_selected == "SOFT":
    b2_test_prob = paper_test_probs.mean(axis=1)
elif b2_selected == "WEIGHTED":
    aps = np.array([
        average_precision_score(y_b2_train.values, b2_oof[0][:, j])
        for j in paper_idx
    ])
    weights = np.maximum(0, aps - y_b2_train.mean())
    weights = (
        np.ones(len(weights)) / len(weights)
        if weights.sum() == 0 else weights / weights.sum()
    )
    b2_test_prob = paper_test_probs @ weights
elif b2_selected == "STACK":
    meta = LogisticRegression(max_iter=5000).fit(
        b2_oof[0][:, paper_idx], y_b2_train.values
    )
    b2_test_prob = meta.predict_proba(paper_test_probs)[:, 1]
elif b2_selected == "HARD":
    votes = (paper_test_probs >= 0.5).sum(axis=1)
    soft = paper_test_probs.mean(axis=1)
    b2_test_prob = np.where(votes > 2, 1.0, np.where(votes < 2, 0.0, soft))

b2_holdout = b2_metric_row(y_b2_test, b2_test_prob)
print("\nB2 hold-out result for the selected model:")
print({k: round(v, 4) for k, v in b2_holdout.items()})


## 6. What B2 found

B2 is the clean estimate. The hold-out was used once, after the CV selection rule was applied.

### Base learners (5 × 5 CV, mean)

| Model | Acc | Balanced | Recall | Macro F1 | ROC | PR | MCC | Brier |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| XGBoost | 87.4% | 67.0% | 36.8% | 70.7% | 81.1% | 59.5% | 45.4% | 10.0 |
| Gradient Boosting | 87.4% | 67.6% | 38.3% | 71.1% | 80.5% | 58.8% | 45.9% | 10.0 |
| Random Forest | 86.6% | 63.0% | 28.1% | 66.3% | 80.7% | 57.2% | 39.5% | 10.6 |
| LightGBM | 86.6% | 64.9% | 32.9% | 68.1% | 79.6% | 55.9% | 40.7% | 11.7 |
| Extra Trees | 85.8% | 61.6% | 25.8% | 64.3% | 79.9% | 53.5% | 35.2% | 10.8 |
| SVM | 85.4% | 67.6% | 41.2% | 69.4% | 78.3% | 52.7% | 40.4% | 11.6 |
| AdaBoost | 83.2% | 72.1% | 55.8% | 70.7% | 80.0% | 57.1% | 41.9% | 17.1 |
| Decision Tree | 79.0% | 62.3% | 37.6% | 62.0% | 68.1% | 30.5% | 24.3% | 16.8 |
| Logistic Regression | 77.6% | 75.0% | 71.3% | 68.1% | 82.5% | 60.0% | 40.7% | 15.1 |

### Ensembles over the four paper learners

| Ensemble | Acc | Balanced | Recall | Macro F1 | ROC | PR | MCC | Brier |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Weighted voting | 87.3% | 66.5% | 35.7% | 70.2% | 81.5% | 58.9% | 44.7% | 9.8 |
| Stacking | 87.1% | 65.9% | 34.6% | 69.5% | 80.9% | 58.2% | 43.3% | 9.8 |
| Soft voting | 86.6% | 65.8% | 34.9% | 69.1% | 80.8% | 56.2% | 41.7% | 10.2 |
| Hard voting | 86.7% | 65.4% | 34.0% | 68.8% | — | — | 41.6% | — |

Hard voting has no probability output, so ROC-AUC, PR-AUC, and Brier are not meaningful and are marked as missing.

Weighted voting fits its weights on the same OOF matrix used to score it, so its estimate is slightly optimistic. Stacking uses a cross-validated meta-learner and is less exposed to this problem. The ensemble comparison should be read with that caveat.

### Selected model and hold-out

Selection: among models with CV accuracy ≥ 85%, XGBoost has the highest PR-AUC.

**XGBoost hold-out result**

| Metric | Value |
|---|---:|
| Accuracy | 87.07% |
| Balanced accuracy | 63.02% |
| Precision | 76.47% |
| Recall | 27.66% |
| F1 | 40.62% |
| Macro F1 | 66.69% |
| ROC-AUC | 79.91% |
| PR-AUC | 53.32% |
| MCC | 40.89% |
| Brier | 0.106 |

The hold-out is lower than the CV estimate, especially for PR-AUC and recall. That is expected with 47 positives and a single 20% split. It is also a warning against reporting only the best CV number.

### Target check

| B2 target | Target range | XGBoost CV | Verdict |
|---|---|---:|---|
| Accuracy | 85–88% | 87.4% | pass |
| Balanced accuracy | 68–72% | 67.0% | just below |
| Recall | 55–65% | 36.8% | fail at 0.5 threshold |
| Macro F1 | 68–72% | 70.7% | pass |
| ROC-AUC | 80–83% | 81.1% | pass |
| PR-AUC | 52–58% | 59.5% | slightly above |
| MCC | 38–45% | 45.4% | slightly above |
| Brier | 11–13% | 10.0% | better than target |

The recall failure is a threshold problem, not necessarily a ranking problem. XGBoost ranks reasonably (ROC 81.1, PR 59.5) but the default 0.5 cut-off is conservative. B3 will tune the threshold for capacity and cost.

### Three findings

1. **A credible clean model reaches about 87% accuracy, not 95%.**  
   XGBoost is at 87.4% CV and 87.1% hold-out. That is the honest range for this dataset under this protocol.

2. **The best PR-AUC model is not the best accuracy model.**  
   Logistic regression has PR-AUC 60.0% and recall 71.3%, but accuracy 77.6% and Brier 15.1. XGBoost has PR-AUC 59.5% and accuracy 87.4%. The choice depends on whether the deployment values ranking, recall, or calibrated accuracy.

3. **The ensembles do not materially improve on the best base learner.**  
   Weighted voting reaches 87.3% accuracy and 58.9% PR-AUC; stacking reaches 87.1% and 58.2%. XGBoost is at 87.4% and 59.5%. The ensemble gain is zero or negative on this dataset. Complexity is not justified by these results.


## 7. B1 versus B2 versus the published studies

### B1 versus B2

| Comparison | B1 | B2 | Difference |
|---|---:|---:|---:|
| Accuracy, all test rows | 92.87% | 87.43% (CV) | **5.44 pp** |
| Accuracy, real test rows only | 88.50% | 87.43% (CV) | **1.07 pp** |
| PR-AUC | 98.30% | 59.50% (CV) | **38.8 pp** |
| Recall | 89.87% | 36.81% (CV) | **53.1 pp** |

The accuracy gap between B1 and B2 is **5.44 pp**, so H1c (`B1a − B2 ≥ 5 pp`) passes narrowly. But the more informative comparison is the real-row accuracy: 88.50% for B1 versus 87.43% for B2. Once synthetic test rows are removed, the two protocols are almost tied.

That is the central audit result:

> The headline 95.0% is inflated mainly by the composition of the test set. The model is not learning a fundamentally better attrition signal; it is being evaluated on easier synthetic rows.

### Comparison with the published frontier

| Source | Model | Acc | Recall | F1 | ROC | PR | Brier |
|---|---|---:|---:|---:|---:|---:|---:|
| Jia (2026) | XGBoost | 85.3% | 33.8% | 42.5% (binary) | 77.3% | 47.9% | 11.9 |
| Căvescu (2026) | XGBoost / LightGBM / LR | — | 45.0% | 43.9% (LR) | 79.1% | 55.6% | — |
| Wang & Liu (2026) | Logistic regression | 86.05% | 34.04% | 70.4% (macro) | 81.2% | 58.4% | — |
| **This work (B2)** | **XGBoost** | **87.4%** | **36.8%** | **70.7% (macro)** | **81.1%** | **59.5%** | **10.0** |
| **This work (B2)** | **Weighted ensemble** | **87.3%** | **35.7%** | **70.2% (macro)** | **81.5%** | **58.9%** | **9.8** |
| **This work (B2)** | **Logistic regression** | 77.6% | **71.3%** | 68.1% (macro) | **82.5%** | **60.0%** | 15.1 |

**Important caveat:** these are not like-for-like comparisons. Jia uses a 7:3 hold-out with inner CV; Căvescu uses a hold-out plus five-fold CV; Wang and Liu use five-fold CV; our B2 uses 5 × 5 repeated CV. F1 is binary for Jia and Căvescu, macro for Wang and for us. Thresholds differ. The table shows context, not a leaderboard.

### Did we improve the results?

**Against the primary paper’s 95.0%: no.**  
Our clean estimate is 87.4% CV and 87.1% hold-out. The 95.0% is not a credible generalization estimate, so “improving” it is the wrong target.

**Against the credible published frontier: marginally, and not decisively.**  
B2 XGBoost is numerically above Jia on accuracy, recall, PR-AUC, ROC-AUC, and Brier, and is close to Wang and Liu. The differences are small relative to the uncertainty of 47 positives. We should not claim a decisive win, especially because the CV protocols differ.

**Against our own B1: yes, in credibility, not in headline accuracy.**  
B2 gives a lower number, but it is the number that means what it says. It also shows that the ensemble complexity does not pay off: XGBoost alone is as good as weighted voting or stacking.

**The defensible contribution is methodological:**  
a measured audit of the 95.0% claim, a clean baseline with intervals, and evidence that the reported ensemble advantage is not robust. That is a stronger paper than another table of inflated accuracies.


## 8. B3 — calibration, thresholds, fairness, SHAP, and transfer

B3 audits the selected B2 model, not the contaminated B1 model.

1. **Calibration:** raw XGBoost scores, Platt scaling, and isotonic regression, fitted on out-of-fold scores and evaluated on the hold-out.
2. **Thresholds:** default 0.5, top-10% and top-20% capacity, recall targets of 60% and 70%, and a cost ratio of 10:1.
3. **Fairness:** subgroup TPR, FPR, PPV, and flag rate by gender, age band, department, and job role, with suppression of subgroups below 10 positives or 10 negatives.
4. **SHAP:** global importance and local explanations for a true positive, a false positive, and a false negative.
5. **Transfer:** IBM → Saudi and Saudi → IBM on a harmonised 13-feature common subset.

The 0.5 threshold is a convention. B3 asks what happens when the threshold is chosen for a real decision instead.


In [ ]:
# B3.1 Calibration and threshold behaviour for the selected XGBoost model.
# b2_test_prob and b2_oof were produced by the B2 section.

xgb_index = b2_model_names.index("XGB")
oof_xgb = b2_oof[0][:, xgb_index]
test_xgb = b2_test_prob
y_oof = y_b2_train.values
y_test = y_b2_test.values


def logit(prob):
    prob = np.clip(prob, 1e-6, 1 - 1e-6)
    return np.log(prob / (1 - prob))


def expected_calibration_error(y_true, prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    total = len(y_true)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (prob >= low) & (prob < high)
        if mask.sum() == 0:
            continue
        gap = abs(y_true[mask].mean() - prob[mask].mean())
        ece += gap * mask.sum() / total
    return ece


def calibration_metrics(y_true, prob):
    return {
        "brier": brier_score_loss(y_true, prob),
        "log_loss": log_loss(y_true, np.clip(prob, 1e-6, 1 - 1e-6)),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "ece": expected_calibration_error(y_true, prob),
    }


platt = LogisticRegression(max_iter=1000).fit(
    logit(oof_xgb).reshape(-1, 1), y_oof
)
platt_test = platt.predict_proba(logit(test_xgb).reshape(-1, 1))[:, 1]

isotonic = IsotonicRegression(out_of_bounds="clip").fit(oof_xgb, y_oof)
isotonic_test = isotonic.predict(test_xgb)

calibration_table = pd.DataFrame([
    {"calibration": "raw", **calibration_metrics(y_test, test_xgb)},
    {"calibration": "Platt", **calibration_metrics(y_test, platt_test)},
    {"calibration": "isotonic", **calibration_metrics(y_test, isotonic_test)},
])
print("Calibration on the hold-out:")
print(calibration_table.round(4).to_string(index=False))


def threshold_at_capacity(prob, capacity):
    return float(np.quantile(prob, 1 - capacity))


def threshold_for_recall(y_true, prob, target_recall):
    grid = np.unique(np.quantile(prob, np.linspace(0, 1, 501)))
    best = None
    for threshold in sorted(grid, reverse=True):
        pred = (prob >= threshold).astype(int)
        if recall_score(y_true, pred, zero_division=0) >= target_recall:
            best = float(threshold)
            break
    return best if best is not None else float(prob.min())


thresholds = {
    "0.5 default": 0.5,
    "top 10% capacity": threshold_at_capacity(oof_xgb, 0.10),
    "top 20% capacity": threshold_at_capacity(oof_xgb, 0.20),
    "recall 60%": threshold_for_recall(y_oof, oof_xgb, 0.60),
    "recall 70%": threshold_for_recall(y_oof, oof_xgb, 0.70),
    "cost 10:1": 1 / (1 + 10),
}

threshold_rows = []
for name, threshold in thresholds.items():
    pred = (test_xgb >= threshold).astype(int)
    threshold_rows.append({
        "threshold_rule": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "flag_rate": pred.mean(),
        "tp": int(((pred == 1) & (y_test == 1)).sum()),
        "fp": int(((pred == 1) & (y_test == 0)).sum()),
        "fn": int(((pred == 0) & (y_test == 1)).sum()),
    })

threshold_table = pd.DataFrame(threshold_rows)
print("\nHold-out behaviour at different thresholds:")
print(threshold_table.round(4).to_string(index=False))

# Use the recall-60% threshold for the fairness audit.
B3_THRESHOLD = thresholds["recall 60%"]
print(f"\nFairness and local explanations use threshold = {B3_THRESHOLD:.4f}")


## 9. Fairness audit

The audit uses out-of-fold predictions on the 1,176 training rows, not the 294-row hold-out. That gives more subgroup observations, while keeping the threshold fixed from the OOF data.

Groups with fewer than 10 positives or 10 negatives are reported descriptively but not used for gap comparisons.


In [ ]:
# B3.2 Subgroup audit at the selected threshold.

fair = pd.DataFrame({
    "y": y_b2_train.values,
    "prob": oof_xgb,
    "pred": (oof_xgb >= B3_THRESHOLD).astype(int),
    "Gender": X_b2_train["Gender"].values,
    "Age": X_b2_train["Age"].values,
    "Department": X_b2_train["Department"].values,
    "JobRole": X_b2_train["JobRole"].values,
})
fair["AgeBand"] = pd.cut(
    fair["Age"], bins=[0, 30, 40, 50, 100],
    labels=["<=30", "31-40", "41-50", "51+"],
)


def subgroup_table(df, attribute):
    rows = []
    for group, part in df.groupby(attribute, observed=True):
        positives = int(part["y"].sum())
        negatives = int(len(part) - positives)
        if positives < 10 or negatives < 10:
            rows.append({
                "group": str(group), "n": len(part),
                "positives": positives, "negatives": negatives,
                "tpr": np.nan, "fpr": np.nan, "ppv": np.nan,
                "flag_rate": np.nan, "status": "suppressed",
            })
            continue
        tp = int(((part["pred"] == 1) & (part["y"] == 1)).sum())
        fn = int(((part["pred"] == 0) & (part["y"] == 1)).sum())
        fp = int(((part["pred"] == 1) & (part["y"] == 0)).sum())
        tn = int(((part["pred"] == 0) & (part["y"] == 0)).sum())
        rows.append({
            "group": str(group), "n": len(part),
            "positives": positives, "negatives": negatives,
            "tpr": tp / (tp + fn) if tp + fn else np.nan,
            "fpr": fp / (fp + tn) if fp + tn else np.nan,
            "ppv": tp / (tp + fp) if tp + fp else np.nan,
            "flag_rate": part["pred"].mean(),
            "status": "ok",
        })
    return pd.DataFrame(rows)


def gap_for(df, attribute, metric):
    table = subgroup_table(df, attribute)
    valid = table[table.status == "ok"]
    if len(valid) < 2:
        return np.nan
    return valid[metric].max() - valid[metric].min()


def bootstrap_gap(df, attribute, metric, n_boot=200, seed=42):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        sample = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9)))
        gap = gap_for(sample, attribute, metric)
        if not np.isnan(gap):
            values.append(gap)
    if not values:
        return (np.nan, np.nan)
    return tuple(np.percentile(values, [2.5, 97.5]))


for attribute in ["Gender", "AgeBand", "Department", "JobRole"]:
    print(f"\n=== {attribute} ===")
    table = subgroup_table(fair, attribute)
    print(table.round(3).to_string(index=False))
    print("TPR gap:", round(gap_for(fair, attribute, "tpr"), 3))
    print("Flag-rate gap:", round(gap_for(fair, attribute, "flag_rate"), 3))


## 10. SHAP explanations

Global SHAP ranks features by mean absolute contribution on the hold-out. Local explanations show the top contributions for three cases: a true positive, a false positive, and a false negative.

SHAP explains the model, not the employee's reason for leaving. A positive contribution is not a causal effect.


In [ ]:
# B3.3 SHAP for the selected XGBoost model.
import shap

shap_pipe = make_b2_pipeline("XGB").fit(X_b2_train, y_b2_train)
shap_prep = shap_pipe.named_steps["prep"]
shap_model = shap_pipe.named_steps["model"]
X_test_transformed = shap_prep.transform(X_b2_test)
feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in shap_prep.get_feature_names_out()
]

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_test_transformed)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
global_importance = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .head(15)
)
print("Global SHAP importance (hold-out):")
print(global_importance.round(4).to_string(index=False))

holdout_pred = (test_xgb >= 0.5).astype(int)
case_indices = {
    "true positive": int(np.where((y_test == 1) & (holdout_pred == 1))[0][0]),
    "false positive": int(np.where((y_test == 0) & (holdout_pred == 1))[0][0]),
    "false negative": int(np.where((y_test == 1) & (holdout_pred == 0))[0][0]),
}

for label, idx in case_indices.items():
    contributions = pd.DataFrame({
        "feature": feature_names,
        "shap": shap_values[idx],
        "value": X_test_transformed[idx],
    })
    contributions["abs_shap"] = contributions["shap"].abs()
    top = contributions.sort_values("abs_shap", ascending=False).head(5)
    print(f"\n{label}: predicted probability = {test_xgb[idx]:.4f}")
    print(top[["feature", "value", "shap"]].round(4).to_string(index=False))


## 11. Bidirectional transfer

The transfer test uses a 13-feature common subset. IBM features are quantile- or band-mapped to the Saudi ordinal coding. This is a construct-level approximation, not a measurement-equivalence claim.

The in-domain reference is five-fold CV on the same common features within each dataset. The transfer gap is in-domain ROC minus cross-dataset ROC.


In [ ]:
# B3.4 Bidirectional transfer on a common 13-feature subset.

def common_ibm(df):
    out = pd.DataFrame(index=df.index)
    out["AgeBand"] = pd.cut(df["Age"], [0, 30, 40, 50, 100], labels=[0, 1, 2, 3]).astype(int)
    out["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})
    out["Distance"] = pd.qcut(df["DistanceFromHome"], 3, labels=[0, 1, 2], duplicates="drop").astype(int)
    out["OverTime"] = df["OverTime"].map({"No": 0, "Yes": 1})
    out["BusinessTravel"] = df["BusinessTravel"].map({
        "Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2,
    })
    out["EnvSat"] = pd.cut(df["EnvironmentSatisfaction"], [0, 2, 3, 4], labels=[0, 1, 2]).astype(int)
    out["JobSat"] = pd.cut(df["JobSatisfaction"], [0, 2, 3, 4], labels=[0, 1, 2]).astype(int)
    out["WorkLife"] = pd.cut(df["WorkLifeBalance"], [0, 2, 3, 4], labels=[0, 1, 2]).astype(int)
    out["Experience"] = pd.qcut(df["TotalWorkingYears"], 3, labels=[0, 1, 2], duplicates="drop").astype(int)
    out["Tenure"] = pd.qcut(df["YearsAtCompany"], 3, labels=[0, 1, 2], duplicates="drop").astype(int)
    out["Training"] = pd.cut(df["TrainingTimesLastYear"], [-1, 0, 2, 4, 10], labels=[0, 1, 2, 3]).astype(int)
    out["Income"] = pd.qcut(df["MonthlyIncome"], 4, labels=[0, 1, 2, 3], duplicates="drop").astype(int)
    out["RecentPromotion"] = (df["YearsSinceLastPromotion"] > 0).astype(int)
    return out


def common_saudi(df):
    out = pd.DataFrame(index=df.index)
    out["AgeBand"] = df["Age"].astype(int)
    out["Gender"] = df["Gender"].astype(int)
    out["Distance"] = df["Distance_to_work"].astype(int)
    out["OverTime"] = df["OverTime"].astype(int)
    out["BusinessTravel"] = df["Business_Travel"].astype(int)
    out["EnvSat"] = df["Environment_Satisfaction"].astype(int)
    out["JobSat"] = df["Job_Satisfaction"].astype(int)
    out["WorkLife"] = df["Work_Live_Balance"].astype(int)
    out["Experience"] = df["Years_Experience"].astype(int)
    out["Tenure"] = df["Years_experience_lastorganization"].astype(int)
    out["Training"] = df["Training_programs_During_last_three_years"].astype(int)
    out["Income"] = df["MonthlySalary"].astype(int)
    out["RecentPromotion"] = df["Get_Deserved_Promotion"].astype(int)
    return out


def transfer_model(X_source, y_source, X_target, y_target):
    model = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1,
        scale_pos_weight=(y_source == 0).sum() / max(1, (y_source == 1).sum()),
    )
    model.fit(X_source, y_source)
    prob = model.predict_proba(X_target)[:, 1]
    return {
        "roc_auc": roc_auc_score(y_target, prob),
        "pr_auc": average_precision_score(y_target, prob),
        "brier": brier_score_loss(y_target, prob),
        "accuracy": accuracy_score(y_target, (prob >= 0.5).astype(int)),
        "recall": recall_score(y_target, (prob >= 0.5).astype(int), zero_division=0),
        "no_skill_pr": float(np.mean(y_target)),
    }


def in_domain_cv(X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, valid_idx in cv.split(X, y):
        model = XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1,
            scale_pos_weight=(y.iloc[train_idx] == 0).sum() / max(1, (y.iloc[train_idx] == 1).sum()),
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        prob = model.predict_proba(X.iloc[valid_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[valid_idx], prob))
    return float(np.mean(scores))


if saudi_path is not None:
    saudi_df = read_csv_with_fallback(saudi_path)
    y_saudi = saudi_df["Attrition"].astype(int)
    X_common_ibm = common_ibm(X_ibm)
    X_common_saudi = common_saudi(saudi_df)
    y_ibm_common = pd.Series(np.asarray(y_ibm).astype(int), index=X_common_ibm.index)

    transfer_rows = [
        {"setting": "IBM in-domain CV", "roc_auc": in_domain_cv(X_common_ibm, y_ibm_common), "pr_auc": np.nan},
        {"setting": "Saudi in-domain CV", "roc_auc": in_domain_cv(X_common_saudi, y_saudi), "pr_auc": np.nan},
        {"setting": "IBM -> Saudi", **transfer_model(X_common_ibm, y_ibm_common, X_common_saudi, y_saudi)},
        {"setting": "Saudi -> IBM", **transfer_model(X_common_saudi, y_saudi, X_common_ibm, y_ibm_common)},
    ]
    transfer_table = pd.DataFrame(transfer_rows)
    print(transfer_table.round(4).to_string(index=False))
else:
    print("Saudi data not found; transfer section skipped.")


## 12. Can the gaps be reduced? Can the explanations be trusted?

The fairness audit found material age and department gaps. The next question is whether a standard mitigation method reduces them, and at what accuracy cost.

The explanation audit asks a different question: does the global SHAP ranking survive a change of training fold, and does it survive a change of dataset?

Both are diagnostics. A failure is a result.


In [ ]:
# B3.5 Fairness mitigation: reweighing and group-specific thresholds.

def reweighing_weights(groups, labels):
    frame = pd.DataFrame({"g": np.asarray(groups), "y": np.asarray(labels)})
    p_g = frame["g"].value_counts(normalize=True)
    p_y = frame["y"].value_counts(normalize=True)
    p_gy = frame.groupby(["g", "y"]).size() / len(frame)
    weights = np.array([
        p_g[g] * p_y[y] / p_gy[(g, y)]
        for g, y in zip(frame["g"], frame["y"])
    ], dtype=float)
    return weights / weights.mean()


def fit_reweighed(attribute):
    groups = fair[attribute].values
    weights = reweighing_weights(groups, y_b2_train.values)

    prep = make_preprocessor(scale_numeric=False).fit(X_b2_train)
    X_train_encoded = prep.transform(X_b2_train)
    X_test_encoded = prep.transform(X_b2_test)

    X_combined, y_combined, synthetic, parent1, parent2 = manual_smote(
        X_train_encoded, y_b2_train.values,
        k=SMOTE_K, ratio=1.0, seed=RANDOM_STATE,
    )
    n_real = len(y_b2_train)
    synthetic_weights = 0.5 * (
        weights[parent1[n_real:]] + weights[parent2[n_real:]]
    )
    combined_weights = np.concatenate([weights, synthetic_weights])

    model = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_combined, y_combined, sample_weight=combined_weights)
    return model.predict_proba(X_test_encoded)[:, 1]


def holdout_gap(y_true, pred, groups):
    frame = pd.DataFrame({"y": np.asarray(y_true), "pred": np.asarray(pred), "group": np.asarray(groups)})
    table = subgroup_table(frame, "group")
    valid = table[table.status == "ok"]
    tpr_gap = valid["tpr"].max() - valid["tpr"].min() if len(valid) > 1 else np.nan
    flag_gap = valid["flag_rate"].max() - valid["flag_rate"].min() if len(valid) > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, pred),
        "mcc": matthews_corrcoef(y_true, pred),
        "tpr_gap": tpr_gap,
        "flag_gap": flag_gap,
    }


def choose_group_thresholds(y_true, prob, groups, target_tpr):
    thresholds = {}
    for group in np.unique(groups):
        mask = np.asarray(groups) == group
        if mask.sum() < 20 or y_true[mask].sum() < 10:
            continue
        grid = np.unique(np.quantile(prob[mask], np.linspace(0, 1, 101)))
        best = None
        for threshold in grid:
            pred = (prob[mask] >= threshold).astype(int)
            tpr = recall_score(y_true[mask], pred, zero_division=0)
            score = abs(tpr - target_tpr)
            if best is None or score < best[0]:
                best = (score, threshold)
        thresholds[group] = best[1]
    return thresholds


age_test = pd.cut(
    X_b2_test["Age"], bins=[0, 30, 40, 50, 100],
    labels=["<=30", "31-40", "41-50", "51+"],
).astype(str).values
gender_test = X_b2_test["Gender"].values

baseline_pred = (test_xgb >= 0.5).astype(int)
target_tpr = recall_score(fair["y"], fair["pred"], zero_division=0)

mitigation_rows = []
for attribute, test_groups in [("AgeBand", age_test), ("Gender", gender_test)]:
    baseline = holdout_gap(y_test, baseline_pred, test_groups)
    baseline.update(attribute=attribute, method="unmitigated")
    mitigation_rows.append(baseline)

    reweighed_prob = fit_reweighed(attribute)
    reweighed_pred = (reweighed_prob >= 0.5).astype(int)
    reweighed = holdout_gap(y_test, reweighed_pred, test_groups)
    reweighed.update(attribute=attribute, method="reweighed")
    mitigation_rows.append(reweighed)

    oof_groups = fair[attribute].astype(str).values
    thresholds = choose_group_thresholds(
        fair["y"].values, fair["prob"].values, oof_groups, target_tpr
    )
    group_pred = np.array([
        int(test_xgb[i] >= thresholds.get(test_groups[i], 0.5))
        for i in range(len(test_groups))
    ])
    group_result = holdout_gap(y_test, group_pred, test_groups)
    group_result.update(attribute=attribute, method="group thresholds")
    mitigation_rows.append(group_result)

mitigation_table = pd.DataFrame(mitigation_rows)
print("Hold-out fairness mitigation comparison (threshold 0.5 unless stated):")
print(mitigation_table[["attribute", "method", "accuracy", "mcc", "tpr_gap", "flag_gap"]].round(3).to_string(index=False))


In [ ]:
# B3.6 SHAP stability across folds and across datasets.
from scipy.stats import spearmanr


def jaccard_top10(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else np.nan


def shap_mean_abs(model, X_matrix):
    values = shap.TreeExplainer(model).shap_values(X_matrix)
    if isinstance(values, list):
        values = values[1]
    return np.abs(np.asarray(values)).mean(axis=0)


# Fold stability on the B2 training set, with preprocessing fixed once.
stability_prep = make_preprocessor(scale_numeric=False).fit(X_b2_train)
X_stability = stability_prep.transform(X_b2_train)
stability_features = [
    name.replace("num__", "").replace("cat__", "")
    for name in stability_prep.get_feature_names_out()
]

folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
fold_top10, fold_rankings = [], []
for train_idx, valid_idx in folds.split(X_stability, y_b2_train):
    fold_model = XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1,
        scale_pos_weight=(y_b2_train.iloc[train_idx] == 0).sum() / max(1, (y_b2_train.iloc[train_idx] == 1).sum()),
    )
    fold_model.fit(X_stability[train_idx], y_b2_train.iloc[train_idx])
    importance = shap_mean_abs(fold_model, X_stability[valid_idx])
    fold_rankings.append(importance)
    fold_top10.append(np.argsort(importance)[-10:])

jaccards = [
    jaccard_top10(fold_top10[i], fold_top10[j])
    for i in range(len(fold_top10)) for j in range(i + 1, len(fold_top10))
]
fold_spearman = [
    spearmanr(fold_rankings[i], fold_rankings[j]).correlation
    for i in range(len(fold_rankings)) for j in range(i + 1, len(fold_rankings))
]
print("Fold SHAP stability:")
print("  top-10 Jaccard mean / min / max:", round(np.mean(jaccards), 3), round(np.min(jaccards), 3), round(np.max(jaccards), 3))
print("  Spearman mean / min:", round(np.mean(fold_spearman), 3), round(np.min(fold_spearman), 3))

# Cross-dataset SHAP on the 13 common features.
saudi_df_for_shap = read_csv_with_fallback(saudi_path)
X_common_ibm_shap = common_ibm(X_ibm)
X_common_saudi_shap = common_saudi(saudi_df_for_shap)
y_common_ibm_shap = pd.Series(np.asarray(y_ibm).astype(int), index=X_common_ibm_shap.index)
y_common_saudi_shap = saudi_df_for_shap["Attrition"].astype(int)

ibm_shap_model = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    random_state=RANDOM_STATE, n_jobs=-1,
    scale_pos_weight=(y_common_ibm_shap == 0).sum() / max(1, (y_common_ibm_shap == 1).sum()),
).fit(X_common_ibm_shap, y_common_ibm_shap)

saudi_shap_model = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    random_state=RANDOM_STATE, n_jobs=-1,
    scale_pos_weight=(y_common_saudi_shap == 0).sum() / max(1, (y_common_saudi_shap == 1).sum()),
).fit(X_common_saudi_shap, y_common_saudi_shap)

ibm_importance = shap_mean_abs(ibm_shap_model, X_common_ibm_shap)
saudi_importance = shap_mean_abs(saudi_shap_model, X_common_saudi_shap)
cross_jaccard = jaccard_top10(
    np.argsort(ibm_importance)[-10:], np.argsort(saudi_importance)[-10:]
)
cross_spearman = spearmanr(ibm_importance, saudi_importance).correlation
print("\nCross-dataset SHAP (13 common features):")
print("  top-10 Jaccard:", round(cross_jaccard, 3))
print("  Spearman rank correlation:", round(cross_spearman, 3))


## 13. What B3 found

### Calibration

| Calibration | Brier | Log loss | ROC-AUC | PR-AUC | ECE |
|---|---:|---:|---:|---:|---:|
| Raw XGBoost | 0.1060 | 0.3719 | 79.91% | 53.32% | 0.069 |
| Platt | 0.1032 | 0.3442 | 79.91% | 53.32% | 0.035 |
| Isotonic | 0.1032 | 0.3798 | 78.97% | 47.98% | 0.033 |

Platt scaling is the useful result: it cuts the expected calibration error in half and improves log loss, without changing the ranking. Isotonic also improves ECE but damages PR-AUC on this small hold-out. The Brier improvement is only **0.0028**, below the pre-registered 0.02 threshold, so **H4 fails on Brier** even though calibration is visibly better.

### Threshold behaviour

| Rule | Threshold | Acc | Precision | Recall | F1 | Flag rate |
|---|---:|---:|---:|---:|---:|---:|
| Default 0.5 | 0.500 | 87.07% | 76.47% | 27.66% | 40.62% | 5.78% |
| Top 10% capacity | 0.408 | 86.39% | 65.22% | 31.91% | 42.86% | 7.82% |
| Top 20% capacity | 0.181 | 80.61% | 41.67% | 53.19% | 46.73% | 20.41% |
| Recall 60% | 0.167 | 80.61% | 42.42% | 59.57% | 49.56% | 22.45% |
| Recall 70% | 0.112 | 77.21% | 38.37% | 70.21% | 49.62% | 29.25% |
| Cost 10:1 | 0.091 | 75.85% | 36.96% | 72.34% | 48.92% | 31.29% |

The default threshold is conservative. If the HR team can review about 22% of employees, recall rises from 27.7% to 59.6%, at the cost of precision falling from 76.5% to 42.4%. This is the real decision trade-off; there is no threshold that maximizes both.

### Fairness at the recall-60% threshold

| Attribute | TPR gap | Flag-rate gap | Main pattern |
|---|---:|---:|---|
| Gender | 0.121 | 0.005 | Female TPR 0.676, male 0.555 |
| Age band | **0.525** | **0.294** | <=30 TPR 0.805, 41–50 TPR 0.280 |
| Department | 0.213 | 0.157 | HR 0.800, R&D 0.587, Sales 0.592 |
| Job role | 0.290 | 0.418 | Small roles suppressed; Sales Representative highest flag rate |

Age is the dominant disparity. The model is much better at identifying younger leavers than employees aged 41–50. The fairness target of a 0.10 TPR gap is not met for any attribute.

Job-role groups with fewer than 10 positives or 10 negatives are suppressed. That includes Healthcare Representative, Manager, Manufacturing Director, and Research Director. The absence of a reported gap for those groups is not evidence of fairness.

### SHAP

Top global features on the hold-out:

1. `OverTime_No`
2. `NumCompaniesWorked`
3. `DistanceFromHome`
4. `EnvironmentSatisfaction`
5. `JobSatisfaction`
6. `BusinessTravel_Travel_Frequently`
7. `StockOptionLevel`
8. `MaritalStatus_Single`
9. `MonthlyIncome`
10. `Age`

Local cases show the expected limitation: a true positive is driven by relationship satisfaction, overtime, and frequent travel; a false positive is driven by overtime and distance in the opposite direction; a false negative is missed because several protective features dominate. SHAP explains the model, not the employee's reason for leaving.

### Bidirectional transfer

| Setting | ROC-AUC | PR-AUC | Accuracy | Recall | No-skill PR |
|---|---:|---:|---:|---:|---:|
| IBM in-domain CV | 0.726 | — | — | — | — |
| Saudi in-domain CV | 0.846 | — | — | — | — |
| IBM → Saudi | 0.583 | 0.505 | 56.81% | 33.53% | 42.42% |
| Saudi → IBM | 0.489 | 0.168 | 56.12% | 40.08% | 16.12% |

Transfer fails the 0.65 ROC target in both directions. The IBM → Saudi direction is weakly better than chance; Saudi → IBM is essentially chance. The IBM transfer gap is 0.143 (within the 0.15 gap target), but the absolute cross-dataset ROC is too low. The Saudi transfer gap is 0.357, which is a clear failure.

This is consistent with the label-construct warning in the report: IBM attrition is a fictional HR record, Saudi attrition is a survey response, and a 13-feature ordinal harmonisation cannot bridge that difference.

### Fairness mitigation at the default threshold

The mitigation comparison is run at the 0.5 threshold, not at the recall-60% threshold. The operating point matters: the age gap is 0.525 at recall 60%, but 0.294 at 0.5. The comparison below holds the threshold fixed at 0.5 to isolate the mitigation method.

| Attribute | Method | Accuracy | MCC | TPR gap | Flag-rate gap |
|---|---|---:|---:|---:|---:|
| Age band | Unmitigated | 87.1% | 0.409 | 0.294 | 0.012 |
| Age band | **Reweighed** | **86.7%** | 0.382 | **0.151** | 0.035 |
| Age band | Group thresholds | 79.9% | 0.366 | 0.508 | 0.115 |
| Gender | Unmitigated | 87.1% | 0.409 | 0.054 | 0.004 |
| Gender | Reweighed | 86.4% | 0.369 | 0.087 | 0.018 |
| Gender | Group thresholds | 80.3% | 0.371 | 0.113 | 0.109 |

Reweighing is the better mitigation:

- It cuts the age TPR gap from **0.294 to 0.151**, roughly in half.
- The accuracy cost is **0.4 pp**.
- It does not close the gap to the 0.10 target.
- It does not help gender; the gender gap is already small, and reweighing slightly increases it.
- Group-specific thresholds fail on the hold-out. They were selected on OOF predictions and did not generalise; the age gap became worse. This is a useful negative result about threshold overfitting.

### SHAP stability

| Analysis | Jaccard | Spearman | Interpretation |
|---|---:|---:|---|
| Across 5 folds | **0.761** (min 0.667, max 1.000) | **0.924** (min 0.907) | Global ranking is stable across folds |
| IBM vs Saudi, 13 common features | 0.667 | **−0.154** | Top-10 overlap is moderate, but the full ranking is not stable |

The fold result passes the 0.60 Jaccard target: the same features remain near the top when the training fold changes.

The cross-dataset result is mixed and should not be overclaimed. Six or seven of the top ten features overlap, but the rank correlation is negative. The explanations are not portable, which is consistent with the transfer failure. A reviewer should read “top-10 overlap” and “rank stability” as different questions; here they disagree.

### B3 target check

| Target | Target | Observed | Verdict |
|---|---:|---:|---|
| Brier improvement | ≥0.02 | 0.0028 | fail |
| ECE improvement | report | 0.069 → 0.035 | improved |
| Recall at useful capacity | 55–65% | 59.6% at 22.4% flag rate | pass |
| Gender TPR gap | ≤0.10 | 0.121 | fail |
| Age TPR gap | ≤0.10 | 0.525 | fail |
| Department TPR gap | ≤0.10 | 0.213 | fail |
| Age gap after reweighing | ≤0.10 | 0.151 | fail, but improved |
| Accuracy cost of reweighing | ≤3 pp | 0.4 pp | pass |
| Transfer ROC, both directions | ≥0.65 | 0.583 / 0.489 | fail |
| Transfer gap | ≤0.15 | 0.143 / 0.357 | mixed |
| SHAP fold stability | Jaccard ≥0.60 | 0.761 | pass |
| Cross-dataset SHAP stability | report / ρ ≥0.50 | −0.154 | fail |

### What B3 means for the paper

B3 turns the project from “a better accuracy table” into a deployment audit:

- Calibration is worth doing, but the Brier gain is small because the raw XGBoost model is already reasonably calibrated.
- Threshold choice is the main lever on recall. A 22% flag rate buys about 60% recall; a 5.8% flag rate buys only 28% recall.
- The model is not fair by age or department. Age is the dominant disparity.
- Reweighing is a partial repair: it halves the age gap at a 0.4 pp accuracy cost, but does not reach the 0.10 target. Group thresholds do not generalise.
- The model is not portable across datasets. The common-feature transfer fails, especially in the Saudi → IBM direction.
- SHAP is stable across folds but not across datasets. The explanation travels less well than the top-10 overlap suggests.

These are exactly the findings an IEEE paper should report. A paper that hides them would be less credible, not more.


## 14. Next: the IEEE paper

The empirical core is now in place:

- B0 arithmetic anchors.
- B1 protocol reconstruction and leakage measurement.
- B2 leakage-safe baseline and ensemble comparison.
- B3 calibration, thresholds, fairness, SHAP, and transfer.

The paper should report:

1. The primary paper's 95.0% as an upper bound under a contaminated test set.
2. B1's seed distribution, not a single number.
3. B2's clean comparison and the negative ensemble result.
4. B3's threshold trade-off, fairness audit, and transfer failure.
5. The reproducibility package: notebook, data checksums, seeds, and environment.

The author list for the paper is:

**Kritarth Prasad, Divyansh Audichya, Pranjal Gaur, Tanay Patel, Md Safwan Areeb**
